# Lesson SIG 3: The Fourier Transform from the Ground Up [SOLUTIONS GUIDE]

**Foundations · Signal Processing for Neural Time Series**

> **SOLUTIONS GUIDE.** Reference implementations with the same test cells the
> student workbook uses. Executed in CI by `tests/unit/test_curriculum.py`.


SIG 1 said what a recording can represent. SIG 2 said what a filter does to it. This
module builds the transform that both were implicitly using, and then spends most
of its length on the two things about it that are routinely got wrong: what sets
frequency resolution, and what happens when a signal does not sit exactly on a
bin.

**What it assumes**

| From | What is used |
|---|---|
| SIG 1 | Sampling rate, Nyquist, and the folding map. |
| SIG 2 | That a filter has a magnitude and a phase response. |

**What it underwrites**

The `psd_by_condition` recipe, and guardrail **G5**, time-frequency window too
long for the effect. Section 2 is where the window length in that guardrail comes
from.


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(3)
FS = 1000.0
print("Environment initialized for Lesson SIG 3")

---

## 1. The transform is a bank of projections

Take $N$ samples $x[n]$. The discrete Fourier transform is

$$X[k] = \sum_{n=0}^{N-1} x[n]\, e^{-2\pi i k n / N}, \qquad k = 0,\dots,N-1$$

Each $X[k]$ is an inner product between the signal and a complex exponential at
$k$ cycles **per window**. Nothing more mysterious than that: it is a bank of
correlations, run at every frequency the window can distinguish.

The reason it is invertible, and the reason the coefficients do not contaminate
each other, is that those exponentials are **orthogonal over the window**:

$$\sum_{n=0}^{N-1} e^{-2\pi i k n/N}\, \overline{e^{-2\pi i l n/N}} =
\begin{cases} N & k = l \\ 0 & k \ne l\end{cases}$$

That orthogonality is exact only for integer $k$, which is to say only for
frequencies completing a whole number of cycles in the window. Section 3 is about
what happens for every other frequency, which is most of them.

In [ ]:
def dft(x: np.ndarray) -> np.ndarray:
    """Discrete Fourier transform, computed directly from the definition.

    Production equivalent: `np.fft.fft`, which computes the same numbers in
    O(N log N) instead of O(N^2). Never write this loop in real code.
    """
    N = len(x)
    n = np.arange(N)
    k = n.reshape(-1, 1)
    return np.exp(-2j * np.pi * k * n / N) @ x

In [ ]:
# --- TEST CELL FOR STEP 1 ---
x_test = rng.standard_normal(64)

# (a) against the library, which is a different algorithm for the same quantity
assert np.allclose(dft(x_test), np.fft.fft(x_test), atol=1e-9), "must match np.fft.fft"

# (b) orthogonality of the basis, checked directly. This is WHY it is invertible.
N = 64
n = np.arange(N)
gram = np.array([[np.vdot(np.exp(2j*np.pi*k*n/N), np.exp(2j*np.pi*l*n/N)) for l in range(N)]
                 for k in range(N)])
assert np.allclose(gram, N * np.eye(N), atol=1e-9), "the basis must be orthogonal over the window"

# (c) Parseval: the transform is a rotation, so it conserves energy.
energy_time = np.sum(np.abs(x_test) ** 2)
energy_freq = np.sum(np.abs(dft(x_test)) ** 2) / N
print(f"energy in time : {energy_time:.6f}")
print(f"energy in freq : {energy_freq:.6f}")
assert np.isclose(energy_time, energy_freq), "Parseval's identity must hold"

# (d) a pure tone at an exact bin lands in exactly one bin, and nowhere else.
k0 = 7
tone = np.cos(2 * np.pi * k0 * n / N)
spec = np.abs(dft(tone))
occupied = np.flatnonzero(spec > 1e-8)
print(f"\nbins occupied by a cosine at exactly bin {k0}: {occupied}")
assert set(occupied.tolist()) == {k0, N - k0}, "an on-bin tone occupies its bin and its mirror"
print("\nStep 1 passed.")

---

## 2. Resolution is set by duration, and by nothing else

The $k$-th coefficient corresponds to $k$ cycles per window, so with $N$ samples
at $f_s$ the window lasts $T = N/f_s$ seconds and the bin spacing is

$$\Delta f = \frac{f_s}{N} = \frac{1}{T}$$

Read that again: resolution depends on the **duration in seconds**, not on the
number of samples and not on the sampling rate. Sampling a one-second record at
30 kHz instead of 1 kHz gives thirty times as many samples and exactly the same
1 Hz resolution. It buys bandwidth, which SIG 1 covered, not resolution.

The consequence people trip over is **zero-padding**. Appending zeros makes $N$
larger without making $T$ larger, so it produces more bins without producing more
information. It interpolates the same underlying spectrum onto a finer grid. Two
tones that a one-second window cannot separate are not separated by padding that
window to ten seconds' worth of samples.

In [ ]:
# --- TEST CELL FOR STEP 2 ---
def two_tone(duration_s, f1, f2, fs=FS):
    t = np.arange(int(fs * duration_s)) / fs
    return np.sin(2 * np.pi * f1 * t) + np.sin(2 * np.pi * f2 * t)

def peak_count(x, n_fft=None):
    """Number of local maxima in the magnitude spectrum, above a floor."""
    mag = np.abs(np.fft.rfft(x, n=n_fft))
    thresh = 0.25 * mag.max()
    return int(np.sum((mag[1:-1] > mag[:-2]) & (mag[1:-1] > mag[2:]) & (mag[1:-1] > thresh)))

F1, F2 = 20.0, 22.0                      # 2 Hz apart
print(f"two tones {F2 - F1:.0f} Hz apart\n")
print(f"{'window':>10} {'df = 1/T':>10} {'N':>7} {'peaks resolved':>16}")
for dur in (0.25, 0.5, 1.0, 2.0):
    x = two_tone(dur, F1, F2)
    print(f"{dur:>9.2f}s {1/dur:>9.1f}Hz {len(x):>7} {peak_count(x):>16}")

assert peak_count(two_tone(0.25, F1, F2)) == 1, "a 0.25 s window cannot resolve 2 Hz"
assert peak_count(two_tone(2.0, F1, F2)) == 2, "a 2 s window can"

# Zero-padding: same duration, thirty-two times the bins, no new information.
short = two_tone(0.25, F1, F2)
padded_bins = len(np.fft.rfft(short, n=32 * len(short)))
print(f"\n0.25 s window, zero-padded to {32 * len(short)} samples:")
print(f"  bins in the spectrum : {padded_bins}  (was {len(np.fft.rfft(short))})")
print(f"  peaks resolved       : {peak_count(short, n_fft=32 * len(short))}")
assert peak_count(short, n_fft=32 * len(short)) == 1, \
    "zero-padding interpolates; it cannot separate what the duration did not resolve"

# And the negative control: sampling faster does not help either.
fast = two_tone(0.25, F1, F2, fs=30000.0)
assert peak_count(fast) == 1, "30x the samples at the same duration resolves nothing extra"
print(f"  same 0.25 s sampled at 30 kHz, {len(fast)} samples: {peak_count(fast)} peak")
print("\nStep 2 passed. Resolution is 1/T. Not 1/N, not fs/N with a bigger N bought")
print("by padding or by sampling faster. Only more seconds.")

---

## 3. Leakage, and why a window is not optional

Section 1's orthogonality held for integer $k$. A real oscillation does not
consult the analysis window before choosing its frequency, so it almost never
completes a whole number of cycles in it. The transform then represents the
discontinuity at the window edge as well as the tone, and the energy that should
have been in one bin appears spread across all of them. That is **spectral
leakage**.

The implicit rectangular window has a very narrow main lobe, which is good, and
sidelobes falling off at only $-13$ dB, which is catastrophic when a strong
low-frequency component sits beside a weak high-frequency one, which is exactly
the shape of every neural spectrum.

A taper trades those against each other: multiply by a smooth window and the
sidelobes drop by orders of magnitude while the main lobe widens. Widening the
main lobe costs resolution, so this is the same bargain as Section 2, paid
knowingly.

In [ ]:
def leakage_floor_db(x: np.ndarray, window: np.ndarray | None = None) -> float:
    """Worst sidelobe level, in dB relative to the peak, away from the peak bin.

    Apply `window` if given, take the magnitude spectrum, find the peak, and
    report the largest magnitude at least 5 bins away from it, in dB below peak.

    Production equivalent: `scipy.signal.get_window` supplies the tapers, and
    `scipy.signal.welch(..., window=...)` applies one for you.
    """
    xs = x if window is None else x * window
    mag = np.abs(np.fft.rfft(xs))
    peak_bin = int(np.argmax(mag))
    mask = np.ones(len(mag), dtype=bool)
    lo = max(0, peak_bin - 5)
    mask[lo:peak_bin + 6] = False
    return float(20 * np.log10(np.max(mag[mask]) / mag[peak_bin]))

In [ ]:
# --- TEST CELL FOR STEP 3 ---
N_1S = 1000                                 # 1 s at 1000 Hz, so bins are 1 Hz apart
t3 = np.arange(N_1S) / FS

on_bin = np.sin(2 * np.pi * 20.0 * t3)    # exactly 20 cycles in the window
off_bin = np.sin(2 * np.pi * 20.5 * t3)   # 20.5 cycles: the worst case

print(f"{'signal':>28} {'worst sidelobe':>16}")
print(f"{'20.0 Hz, rectangular':>28} {leakage_floor_db(on_bin):>13.1f} dB")
print(f"{'20.5 Hz, rectangular':>28} {leakage_floor_db(off_bin):>13.1f} dB")
for name, win in (("Hann", np.hanning(N_1S)), ("Blackman", np.blackman(N_1S))):
    print(f"{'20.5 Hz, ' + name:>28} {leakage_floor_db(off_bin, win):>13.1f} dB")

assert leakage_floor_db(on_bin) < -100, "an on-bin tone barely leaks at all"
assert leakage_floor_db(off_bin) > -40, "a half-bin offset leaks badly with no taper"
assert leakage_floor_db(off_bin, np.hanning(N_1S)) < leakage_floor_db(off_bin) - 20, \
    "a Hann taper must cut the sidelobes by at least 20 dB"

# The cost of the taper, measured: the main lobe gets wider, so resolution drops.
def main_lobe_bins(win, pad=32):
    """Width at -6 dB, in units of the UNPADDED bin spacing.

    Padded so the width is measured with sub-bin precision: integer bin counts
    are too coarse to separate a Hann lobe from a Blackman one.
    """
    sig = win * np.sin(2 * np.pi * 100.0 * np.arange(len(win)) / FS)
    mag = np.abs(np.fft.rfft(sig, n=pad * len(win)))
    return float(np.sum(mag > 0.5 * mag.max())) / pad

print(f"\n{'window':>12} {'main lobe width (bins at -6 dB)':>34}")
for name, win in (("rectangular", np.ones(N_1S)), ("Hann", np.hanning(N_1S)),
                  ("Blackman", np.blackman(N_1S))):
    print(f"{name:>12} {main_lobe_bins(win):>30.2f}")
assert main_lobe_bins(np.blackman(N_1S)) > main_lobe_bins(np.hanning(N_1S)) > main_lobe_bins(np.ones(N_1S)), \
    "lower sidelobes are bought with a wider main lobe, every time"

# Why it matters here: a weak high-frequency component beside a strong slow one.
strong_slow = 100.0 * np.sin(2 * np.pi * 3.5 * t3)
weak_fast = 1.0 * np.sin(2 * np.pi * 80.0 * t3)
def visible(sig, win):
    mag = np.abs(np.fft.rfft(sig * win))
    return float(mag[80] / np.median(mag[60:100]))
print(f"\n80 Hz component visibility beside a 100x larger 3.5 Hz component:")
print(f"  rectangular : {visible(strong_slow + weak_fast, np.ones(N_1S)):.2f}")
print(f"  Hann        : {visible(strong_slow + weak_fast, np.hanning(N_1S)):.2f}")
assert visible(strong_slow + weak_fast, np.hanning(N_1S)) > \
       2 * visible(strong_slow + weak_fast, np.ones(N_1S)), \
    "leakage from the slow component buries the fast one without a taper"
print("\nStep 3 passed. Every neural spectrum has a strong slow component beside")
print("weak fast ones, so the taper is not a refinement. It decides what you can see.")

---

## 4. What you established

1. The DFT is a bank of projections onto complex exponentials that are exactly
   orthogonal over the window. You verified the orthogonality directly, and
   Parseval, before comparing anything to a library.
2. Frequency resolution is $1/T$ in seconds. Zero-padding a 0.25 s window to
   32 times the samples produces 32 times the bins and still cannot separate two
   tones 2 Hz apart. Neither can sampling the same 0.25 s at 30 kHz. Only more
   seconds.
3. A tone half a bin off centre leaks catastrophically with a rectangular window.
   A Hann taper cuts the worst sidelobe by tens of dB and pays for it with a
   wider main lobe. Beside a component 100 times larger, that difference decides
   whether the smaller one is visible at all.

### Exercises

**Exercise 1.** Guardrail G5 refuses a time-frequency window too long for the
effect it claims to measure. Using $\Delta f = 1/T$, work out the shortest window
that can resolve a 4 Hz-wide beta peak, and then the longest window that can
still localise a burst lasting 200 ms. Is there a window length satisfying both?
State what you would do when there is not.

**Exercise 2.** Section 3 measured leakage for a tone exactly half a bin off.
Sweep the offset from 0 to 1 bin in steps of 0.05 and plot the worst sidelobe.
Where is it worst, and what does that say about reporting a peak frequency read
off an unwindowed spectrum?

**Exercise 3.** Zero-padding adds no resolution but it is not useless. Show that
it improves the accuracy with which you can locate a peak's frequency, and
explain why that is not a contradiction.

---

**Next: SIG 4, Welch's method and multitaper estimation.** This module transformed a
signal. SIG 4 asks the harder question of how to estimate a spectrum from a noisy
one, and starts with the fact that the obvious estimator does not improve as you
collect more data.
